#### --- 1. KÜTÜPHANELER ---

In [ ]:
import pandas as pd 
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns 
%matplotlib inline 

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score , confusion_matrix , classification_report
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV , StratifiedKFold
from sklearn.preprocessing import RobustScaler

#### --- 2. VERİ YÜKLEME VE TEMİZLİK ---

📚 Veri setinin ana özellikleri şunlardır:

1-id: Her hastanın benzersiz bir kimliğini temsil eder.

2-diagnosis: Kanserin türünü belirtir. Bu özellik "M" (Malignant - Kötü Huylu) veya "B" (Benign - İyi Huylu) değerlerini alabilir.

3-radius_mean, texture_mean, perimeter_mean, area_mean, smoothness_mean, compactness_mean, concavity_mean, concave points_mean: Kanserin görsel özelliklerinin ortalama değerlerini temsil eder.

In [ ]:
df = pd.read_csv("Cancer_Data.csv")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df = df.drop(["id", "Unnamed: 32"],axis=1)

In [ ]:
df['diagnosis'] = df['diagnosis'].map({'M' : 1 , 'B' : 0})

####  --- 3. EDA (KEŞİFSEL VERİ ANALİZİ) VE KORELASYON ---

In [ ]:
sns.scatterplot(x=df["compactness_se"],y=df["perimeter_worst"],hue =df["diagnosis"])
plt.show()

In [ ]:
sns.scatterplot(x=df["concave points_mean"],y=df["concave points_worst"],hue =df["diagnosis"])
plt.show()

In [ ]:
corr = df.corr()

mask = np.triu(np.ones_like(corr , dtype= bool))

plt.figure(figsize=(20,15))
sns.heatmap(data = corr , mask= mask , annot = True , cmap = "RdYlGn" , fmt=".2f" , center = 0)
plt.show()

#### --- 4. ÇOKLU BAĞLANTI (MULTICOLLINEARITY) VE VIF ANALİZİ ---

In [ ]:
# Birbirini tekrar eden, modele ekstra bilgi katmayan ve algoritmanın kafasını karıştıran değişkenleri bulmak için böyle bir yaklaşım uyguladım.
from statsmodels.stats.outliers_influence import variance_inflation_factor

X = df.drop('diagnosis',axis = 1 )

# X: Bağımsız değişkenlerin olduğu DataFrame
def calculate_vif(X):
    vif_data = pd.DataFrame()
    vif_data["feature"] = X.columns
    vif_data["VIF"] = [variance_inflation_factor(X.values, i) for i in range(len(X.columns))]
    return vif_data

print(calculate_vif(df))

In [ ]:
corr_matrix = df.corr().abs()
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

corr_series = upper.unstack().dropna()
sorted_corr = corr_series.sort_values(ascending=False)

high_corr = sorted_corr[sorted_corr > 0.95]
print("Yüksek Korelasyonlu Değişken Çiftleri:")
print(high_corr)

In [ ]:
# Korelasyon matrisinin mutlak değerini al
corr_matrix = df.drop('diagnosis', axis=1).corr().abs()

# Sadece üst üçgeni seç (aynı çiftleri tekrar işlememek için)
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))

# Korelasyonu 0.95'dan büyük olan kolonları bul
to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]

# Yüksek korelasyonlu sütunları düşürerek "df_reduced" adında yeni bir dataframe oluşturuyoruz.
df_reduced = df.drop(columns=to_drop)

#Çıkartılan veriler 
print(to_drop)

In [ ]:
df_reduced.head()

#### --- 5. VERİ HAZIRLIĞI VE MODELLEME (1. DENEY) ---

In [ ]:
X = df_reduced.drop('diagnosis',axis =1 )
y = df_reduced['diagnosis']

In [ ]:
X_train, X_test, y_train , y_test = train_test_split(X , y , test_size = 0.25 , random_state = 42)

In [ ]:
scaler = RobustScaler()

In [ ]:
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

#### --- 6. NAIVE BAYES MODELİ ---

In [ ]:
model = GaussianNB()

In [ ]:
model.fit(X_train_scaled , y_train)

In [ ]:
y_pred = model.predict(X_test_scaled)

In [ ]:
print("=" * 10)
print("Accuracy Score : ",accuracy_score(y_test , y_pred))
print("=" * 10)
print("Confusion Matrix :\n",confusion_matrix(y_test , y_pred))
print("=" * 10)
print("Classification Report:\n",classification_report(y_test , y_pred))
print("=" * 10)

### ================== Yüksek Korelasyonları silmeden Tekrar Ölçeceğim ======================

In [ ]:
df = pd.read_csv("Cancer_Data.csv")

In [ ]:
df.head()

In [ ]:
df = df.drop(['id', 'Unnamed: 32'], axis=1)

In [ ]:
df['diagnosis'] = df['diagnosis'].map({'M':1,'B':0})

In [ ]:
X = df.drop('diagnosis',axis=1)
y = df['diagnosis']

In [ ]:
X_train , X_test, y_train , y_test = train_test_split(X, y , test_size=0.2 , random_state=42)

In [ ]:
scaler = RobustScaler()

In [ ]:
X_train_scaled= scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
model = GaussianNB()

In [ ]:
model.fit(X_train_scaled , y_train)

In [ ]:
y_pred = model.predict(X_test_scaled)

In [ ]:
print("=" * 10)
print("Accuracy Score : ",accuracy_score(y_test , y_pred))
print("=" * 10)
print("Confusion Matrix :\n",confusion_matrix(y_test , y_pred))
print("=" * 10)
print("Classification Report:",classification_report(y_test , y_pred))
print("=" * 10)